In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import zscore
import matplotlib.pyplot as plt

In [2]:
# ============================================================================
# STEP 1: GENERATE SYNTHETIC DATA
# ============================================================================

print("="*80)
print("STEP 1: GENERATING SYNTHETIC DATA")
print("="*80)

np.random.seed(42)

n_samples = 1000
data = {
    'timestamp': pd.date_range(start='2024-01-01', periods=n_samples, freq='h'),
    'cpu_usage': np.random.normal(50, 10, n_samples),
    'memory_usage': np.random.normal(60, 15, n_samples),
    'network_latency': np.random.normal(100, 20, n_samples),
    'disk_io': np.random.normal(75, 10, n_samples),
    'error_rate': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
}

df = pd.DataFrame(data)
print(f"✓ Generated {len(df)} samples")

STEP 1: GENERATING SYNTHETIC DATA
✓ Generated 1000 samples


In [3]:
# ============================================================================
# STEP 2: DEFINE HELPER FUNCTIONS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: DEFINING HELPER FUNCTIONS")
print("="*80)

def detect_anomalies(data):
    """
    Detect anomalies using Isolation Forest.
    
    Parameters:
    -----------
    data : DataFrame or array-like
        Numeric data only
    
    Returns:
    --------
    array : Anomaly predictions (-1 for anomaly, 1 for normal)
    """
    model = IsolationForest(contamination=0.05, random_state=42)
    model.fit(data)
    anomalies = model.predict(data)
    return anomalies

def root_cause_analysis(X_train, y_train, X_test):
    """
    Train decision tree for root cause analysis.
    
    Parameters:
    -----------
    X_train : DataFrame
        Training features (numeric only)
    y_train : Series
        Training labels
    X_test : DataFrame
        Test features (numeric only)
    
    Returns:
    --------
    model : Trained DecisionTreeClassifier
    predictions : Array of predictions
    """
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return model, predictions

def identify_root_cause(row, feature_importance_df, threshold_percentile=75):
    """
    Identify the root cause category based on anomalous metrics.
    
    Parameters:
    -----------
    row : Series
        Single row from dataframe
    feature_importance_df : DataFrame
        Feature importance ranking
    threshold_percentile : float
        Percentile threshold for anomaly detection
    
    Returns:
    --------
    str : Root cause category
    """
    # Check each metric against thresholds
    cpu_threshold = df['cpu_usage'].quantile(threshold_percentile / 100)
    memory_threshold = df['memory_usage'].quantile(threshold_percentile / 100)
    network_threshold = df['network_latency'].quantile(threshold_percentile / 100)
    disk_threshold = df['disk_io'].quantile(threshold_percentile / 100)
    
    # Identify which metrics are anomalous
    if row['network_latency'] > network_threshold * 2:  # Significantly high
        return "network_error"
    elif row['cpu_usage'] > cpu_threshold * 1.5:
        return "high_cpu_usage"
    elif row['memory_usage'] > memory_threshold * 1.5:
        return "high_memory_usage"
    elif row['disk_io'] < disk_threshold * 0.5:  # Low disk I/O
        return "disk_issue"
    elif row['error_rate'] == 1:
        return "application_error"
    else:
        # Use top feature from importance ranking
        top_feature = feature_importance_df.iloc[0]['feature']
        
        if 'network' in top_feature:
            return "network_error"
        elif 'cpu' in top_feature:
            return "high_cpu_usage"
        elif 'memory' in top_feature:
            return "high_memory_usage"
        elif 'disk' in top_feature:
            return "disk_issue"
        elif 'error' in top_feature:
            return "application_error"
        else:
            return "unknown_issue"

def recommend_solution(root_cause):
    """
    Recommend solution based on identified root cause.
    
    Parameters:
    -----------
    root_cause : str
        Identified root cause category
    
    Returns:
    --------
    str : Recommended solution
    """
    solutions = {
        "network_error": """
        🌐 NETWORK ERROR DETECTED
        
        Recommended Actions:
        1. Restart the network service
        2. Check network interface status: ifconfig or ip addr
        3. Verify DNS resolution: nslookup google.com
        4. Test connectivity: ping gateway
        5. Check for network congestion or bandwidth issues
        6. Review firewall rules and security groups
        7. Inspect network cable connections (if physical)
        
        Commands:
        - sudo systemctl restart networking
        - sudo service network-manager restart
        - traceroute <destination> (to identify bottleneck)
        """,
        
        "database_issue": """
        💾 DATABASE ISSUE DETECTED
        
        Recommended Actions:
        1. Check the database connection and restart the service
        2. Verify database credentials and permissions
        3. Check database server status
        4. Review connection pool settings
        5. Check for long-running queries
        6. Verify disk space on database server
        
        Commands:
        - sudo systemctl restart postgresql (or mysql/mongodb)
        - psql -U username -d database -c "SELECT 1" (test connection)
        - Check logs: /var/log/postgresql/ or /var/log/mysql/
        """,
        
        "high_cpu_usage": """
        🔥 HIGH CPU USAGE DETECTED
        
        Recommended Actions:
        1. Optimize running processes or allocate more resources
        2. Identify CPU-intensive processes: top or htop
        3. Kill unnecessary processes
        4. Check for infinite loops or runaway scripts
        5. Scale horizontally (add more instances)
        6. Optimize application code
        
        Commands:
        - top -o %CPU (sort by CPU usage)
        - ps aux --sort=-%cpu | head
        - kill -9 <PID> (to terminate process)
        - nice -n 19 <command> (lower priority)
        """,
        
        "high_memory_usage": """
        💾 HIGH MEMORY USAGE DETECTED
        
        Recommended Actions:
        1. Identify memory-intensive processes
        2. Clear cache and buffers
        3. Restart memory-heavy services
        4. Check for memory leaks
        5. Increase swap space temporarily
        6. Add more RAM if persistent
        
        Commands:
        - free -h (check memory usage)
        - top -o %MEM (sort by memory)
        - sudo sync; echo 3 > /proc/sys/vm/drop_caches (clear cache)
        - ps aux --sort=-%mem | head
        """,
        
        "disk_issue": """
        💽 DISK I/O ISSUE DETECTED
        
        Recommended Actions:
        1. Check disk space: df -h
        2. Identify I/O intensive processes: iotop
        3. Clear unnecessary files and logs
        4. Check disk health: smartctl
        5. Optimize database queries if database-heavy
        6. Consider SSD upgrade or RAID configuration
        
        Commands:
        - df -h (disk space)
        - sudo iotop (I/O monitoring)
        - du -sh /* | sort -h (find large directories)
        - sudo smartctl -a /dev/sda (disk health)
        """,
        
        "application_error": """
        ⚠️ APPLICATION ERROR DETECTED
        
        Recommended Actions:
        1. Check application logs for error messages
        2. Restart the application service
        3. Verify application configuration
        4. Check for recent code deployments
        5. Review error tracking systems (Sentry, Rollbar, etc.)
        6. Rollback to previous stable version if needed
        
        Commands:
        - sudo systemctl restart <service-name>
        - tail -f /var/log/application.log
        - journalctl -u <service-name> -f
        """,
        
        "unknown_issue": """
        ❓ UNKNOWN ISSUE DETECTED
        
        Recommended Actions:
        1. Review all system metrics holistically
        2. Check system logs: /var/log/syslog or journalctl
        3. Monitor system for patterns
        4. Collect diagnostic information
        5. Contact support with detailed logs
        
        Commands:
        - dmesg | tail (kernel messages)
        - journalctl -xe (system logs)
        - vmstat 1 (system statistics)
        """
    }
    
    return solutions.get(root_cause, """
    ℹ️ NO SPECIFIC RECOMMENDATION AVAILABLE
    
    General Troubleshooting Steps:
    1. Review system logs
    2. Check resource utilization (CPU, Memory, Disk, Network)
    3. Restart affected services
    4. Contact system administrator for assistance
    """)

print("✓ Helper functions defined")


STEP 2: DEFINING HELPER FUNCTIONS
✓ Helper functions defined


In [4]:
# ============================================================================
# STEP 3: INITIAL ANOMALY DETECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 3: INITIAL ANOMALY DETECTION")
print("="*80)

# Extract only numeric columns
numeric_data = df.select_dtypes(include=[np.number])
print(f"Numeric columns: {numeric_data.columns.tolist()}")

# Detect anomalies
df['anomaly'] = detect_anomalies(numeric_data)

print(f"\nAnomalies detected: {(df['anomaly'] == -1).sum()}")
print(f"Normal instances: {(df['anomaly'] == 1).sum()}")


STEP 3: INITIAL ANOMALY DETECTION
Numeric columns: ['cpu_usage', 'memory_usage', 'network_latency', 'disk_io', 'error_rate']

Anomalies detected: 50
Normal instances: 950


In [5]:
# ============================================================================
# STEP 4: TRAIN ROOT CAUSE MODEL
# ============================================================================

print("\n" + "="*80)
print("STEP 4: TRAINING ROOT CAUSE MODEL")
print("="*80)

# Prepare training data with only numeric columns
X_train = df.select_dtypes(include=[np.number]).drop('anomaly', axis=1)
y_train = df['anomaly']

print(f"Training features: {X_train.columns.tolist()}")
print(f"Training samples: {len(X_train)}")

# Train model
initial_model = DecisionTreeClassifier(random_state=42)
initial_model.fit(X_train, y_train)

print("✓ Model trained successfully")


STEP 4: TRAINING ROOT CAUSE MODEL
Training features: ['cpu_usage', 'memory_usage', 'network_latency', 'disk_io', 'error_rate']
Training samples: 1000
✓ Model trained successfully


In [6]:
# ============================================================================
# STEP 5: SIMULATE NETWORK ERROR
# ============================================================================

print("\n" + "="*80)
print("STEP 5: SIMULATING NETWORK ERROR")
print("="*80)

# Store original value
original_latency = df.loc[0, 'network_latency']

# Simulate network error
df.loc[0, 'network_latency'] = 1000  # Simulating high network latency

print(f"✓ Simulated high network latency at index 0")
print(f"  Original: {original_latency:.2f}ms")
print(f"  Simulated: 1000ms (10x normal)")


STEP 5: SIMULATING NETWORK ERROR
✓ Simulated high network latency at index 0
  Original: 86.50ms
  Simulated: 1000ms (10x normal)


In [7]:
# ============================================================================
# STEP 6: RUN TROUBLESHOOTING AGENT
# ============================================================================

print("\n" + "="*80)
print("STEP 6: RUNNING TROUBLESHOOTING AGENT")
print("="*80)

# Re-detect anomalies on updated data
numeric_data_updated = df.select_dtypes(include=[np.number])
anomalies = detect_anomalies(numeric_data_updated)
df['anomaly'] = anomalies

# Prepare test data
X_test = df.select_dtypes(include=[np.number]).drop('anomaly', axis=1, errors='ignore')

print(f"Test features: {X_test.columns.tolist()}")

# Run root cause analysis
model, predicted_causes = root_cause_analysis(X_train, y_train, X_test)

# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)


STEP 6: RUNNING TROUBLESHOOTING AGENT
Test features: ['cpu_usage', 'memory_usage', 'network_latency', 'disk_io', 'error_rate']


In [8]:
# ============================================================================
# STEP 7: IDENTIFY ROOT CAUSE AND GET RECOMMENDATION
# ============================================================================

print("\n" + "="*80)
print("STEP 7: ROOT CAUSE IDENTIFICATION")
print("="*80)

# Get the first row (simulated error)
error_row = df.loc[0]

# Identify root cause
root_cause = identify_root_cause(error_row, feature_importance)

print(f"\n✓ Root Cause Identified: {root_cause}")
print(f"  Anomaly Status: {'ANOMALY DETECTED' if predicted_causes[0] == -1 else 'NORMAL'}")

# Get solution recommendation
solution = recommend_solution(root_cause)


STEP 7: ROOT CAUSE IDENTIFICATION

✓ Root Cause Identified: network_error
  Anomaly Status: NORMAL


In [9]:
# ============================================================================
# STEP 8: DISPLAY COMPREHENSIVE TROUBLESHOOTING REPORT
# ============================================================================

print("\n" + "="*80)
print("TROUBLESHOOTING AGENT - COMPREHENSIVE REPORT")
print("="*80)

print(f"\n📅 Timestamp: {error_row['timestamp']}")
print(f"🔍 Detected Issue: {root_cause.replace('_', ' ').title()}")
print(f"🚨 Anomaly Prediction: {predicted_causes[0]} ({'ANOMALY' if predicted_causes[0] == -1 else 'NORMAL'})")

print("\n" + "-"*80)
print("CURRENT METRICS")
print("-"*80)
print(f"CPU Usage:        {error_row['cpu_usage']:.2f}%")
print(f"Memory Usage:     {error_row['memory_usage']:.2f}%")
print(f"Network Latency:  {error_row['network_latency']:.2f}ms ⚠️ CRITICAL!")
print(f"Disk I/O:         {error_row['disk_io']:.2f} MB/s")
print(f"Error Rate:       {error_row['error_rate']}")

print("\n" + "-"*80)
print("BASELINE COMPARISON")
print("-"*80)
print(f"Normal CPU Usage:        {df['cpu_usage'].median():.2f}%")
print(f"Normal Memory Usage:     {df['memory_usage'].median():.2f}%")
print(f"Normal Network Latency:  {df['network_latency'].median():.2f}ms")
print(f"Normal Disk I/O:         {df['disk_io'].median():.2f} MB/s")

print("\n" + "-"*80)
print("DEVIATION ANALYSIS")
print("-"*80)
deviation_cpu = ((error_row['cpu_usage'] - df['cpu_usage'].median()) / df['cpu_usage'].median() * 100)
deviation_memory = ((error_row['memory_usage'] - df['memory_usage'].median()) / df['memory_usage'].median() * 100)
deviation_network = ((error_row['network_latency'] - df['network_latency'].median()) / df['network_latency'].median() * 100)
deviation_disk = ((error_row['disk_io'] - df['disk_io'].median()) / df['disk_io'].median() * 100)

print(f"CPU Usage Deviation:       {deviation_cpu:+.2f}%")
print(f"Memory Usage Deviation:    {deviation_memory:+.2f}%")
print(f"Network Latency Deviation: {deviation_network:+.2f}% ⚠️")
print(f"Disk I/O Deviation:        {deviation_disk:+.2f}%")

print("\n" + "-"*80)
print("ROOT CAUSE RANKING")
print("-"*80)
print(feature_importance.to_string(index=False))

print("\n" + "="*80)
print("RECOMMENDED SOLUTION")
print("="*80)
print(solution)


TROUBLESHOOTING AGENT - COMPREHENSIVE REPORT

📅 Timestamp: 2024-01-01 00:00:00
🔍 Detected Issue: Network Error
🚨 Anomaly Prediction: 1 (NORMAL)

--------------------------------------------------------------------------------
CURRENT METRICS
--------------------------------------------------------------------------------
CPU Usage:        54.97%
Memory Usage:     80.99%
Network Latency:  1000.00ms ⚠️ CRITICAL!
Disk I/O:         55.92 MB/s
Error Rate:       0

--------------------------------------------------------------------------------
BASELINE COMPARISON
--------------------------------------------------------------------------------
Normal CPU Usage:        50.25%
Normal Memory Usage:     60.95%
Normal Network Latency:  100.01ms
Normal Disk I/O:         75.00 MB/s

--------------------------------------------------------------------------------
DEVIATION ANALYSIS
--------------------------------------------------------------------------------
CPU Usage Deviation:       +9.38%
Mem

In [10]:
# ============================================================================
# STEP 9: DEMONSTRATE WITH DIFFERENT ROOT CAUSES
# ============================================================================

print("\n" + "="*80)
print("STEP 9: TESTING ALL SOLUTION RECOMMENDATIONS")
print("="*80)

# Test all possible root causes
test_causes = ["network_error", "database_issue", "high_cpu_usage", 
               "high_memory_usage", "disk_issue", "application_error", "unknown_issue"]

for cause in test_causes:
    print(f"\n{'='*80}")
    print(f"TESTING: {cause.upper()}")
    print(f"{'='*80}")
    solution_test = recommend_solution(cause)
    print(solution_test)



STEP 9: TESTING ALL SOLUTION RECOMMENDATIONS

TESTING: NETWORK_ERROR

        🌐 NETWORK ERROR DETECTED
        
        Recommended Actions:
        1. Restart the network service
        2. Check network interface status: ifconfig or ip addr
        3. Verify DNS resolution: nslookup google.com
        4. Test connectivity: ping gateway
        5. Check for network congestion or bandwidth issues
        6. Review firewall rules and security groups
        7. Inspect network cable connections (if physical)
        
        Commands:
        - sudo systemctl restart networking
        - sudo service network-manager restart
        - traceroute <destination> (to identify bottleneck)
        

TESTING: DATABASE_ISSUE

        💾 DATABASE ISSUE DETECTED
        
        Recommended Actions:
        1. Check the database connection and restart the service
        2. Verify database credentials and permissions
        3. Check database server status
        4. Review connection pool settings

In [11]:
# ============================================================================
# STEP 10: SAVE COMPREHENSIVE REPORT
# ============================================================================

print("\n" + "="*80)
print("STEP 10: SAVING COMPREHENSIVE REPORT")
print("="*80)

# Create detailed report
report = pd.DataFrame({
    'timestamp': [error_row['timestamp']],
    'root_cause': [root_cause],
    'anomaly_detected': ['Yes' if predicted_causes[0] == -1 else 'No'],
    'cpu_usage_pct': [error_row['cpu_usage']],
    'cpu_deviation_pct': [deviation_cpu],
    'memory_usage_pct': [error_row['memory_usage']],
    'memory_deviation_pct': [deviation_memory],
    'network_latency_ms': [error_row['network_latency']],
    'network_deviation_pct': [deviation_network],
    'disk_io_mbps': [error_row['disk_io']],
    'disk_deviation_pct': [deviation_disk],
    'error_rate': [error_row['error_rate']],
    'top_contributing_feature': [feature_importance.iloc[0]['feature']],
    'recommendation_summary': ['Restart network service - see full report']
})

report.to_csv('troubleshooting_report.csv', index=False)
print("✓ Report saved to 'troubleshooting_report.csv'")

# Save feature importance
feature_importance.to_csv('feature_importance.csv', index=False)
print("✓ Feature importance saved to 'feature_importance.csv'")

# Save full dataset with anomalies
df.to_csv('full_dataset_with_anomalies.csv', index=False)
print("✓ Full dataset saved to 'full_dataset_with_anomalies.csv'")

# Save detailed solution recommendations
solutions_df = pd.DataFrame({
    'root_cause': test_causes,
    'recommendation': [recommend_solution(cause) for cause in test_causes]
})
solutions_df.to_csv('solution_recommendations.csv', index=False)
print("✓ Solution recommendations saved to 'solution_recommendations.csv'")

print("\n" + "="*80)
print("✅ TROUBLESHOOTING AGENT COMPLETE!")
print("="*80)
print(f"\nSummary:")
print(f"  Root Cause: {root_cause}")
print(f"  Status: {'ANOMALY DETECTED' if predicted_causes[0] == -1 else 'SYSTEM NORMAL'}")
print(f"  Primary Issue: Network latency {error_row['network_latency']:.0f}ms (normal: ~{df['network_latency'].median():.0f}ms)")
print(f"  Recommendation: {root_cause.replace('_', ' ').title()} - Check saved report for details")


STEP 10: SAVING COMPREHENSIVE REPORT
✓ Report saved to 'troubleshooting_report.csv'
✓ Feature importance saved to 'feature_importance.csv'
✓ Full dataset saved to 'full_dataset_with_anomalies.csv'
✓ Solution recommendations saved to 'solution_recommendations.csv'

✅ TROUBLESHOOTING AGENT COMPLETE!

Summary:
  Root Cause: network_error
  Status: SYSTEM NORMAL
  Primary Issue: Network latency 1000ms (normal: ~100ms)
  Recommendation: Network Error - Check saved report for details
